# Post-blending: run-7 ⊕ run-5

**Why.** The two models agree only moderately (daily rank correlation ≈ 0.79,
N-29), so they carry partly different signals. Averaging two different, similarly
good predictions usually keeps the strengths of both and dilutes their separate
errors.

**The transformation** (section 2):
`blend = w × (this run − its day mean) + (1 − w) × (partner − its day mean)`.
- Both inputs are demeaned per day first, the standard output since N-24.
- `w = 0` is the partner alone (demeaned); `w = 1` is this run alone.

**Choosing w without peeking** (section 3):
- Each window's weight is chosen on **that window's validation predictions**: the
  purged k-fold predictions of its history, made before its test period.
- Two rules: the weight with the best validation R² (magnitudes), and the one with
  the best validation Rank IC (ranking).
- One weight fitted on all windows' validation data would use history *after* the
  earlier test windows, so it isn't used for the verdict.

**What can't change:** nothing about either model. Only how the two outputs are
combined.

**Sections:** 1. setup · 2. the transformation · 3. choosing w on validation ·
4. test results · 5. hindsight scan on test · 6. findings · 7. summary.


## 1. Setup


In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_width_chars(250)

RUN_DIR = Path("/workspaces/TimeSeries48/runs/run-7_single-day-rawfeat+rank+std-p5fcv")
PARTNER_DIR = Path("/workspaces/TimeSeries48/runs/run-5_baseline-all-feat-purged-5-fold-val-retrain")
PLOTS_DIR = RUN_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
WEIGHTS = np.round(np.arange(0, 1.0001, 0.05), 2)  # w on this run; 1 - w on the partner.
NW_LAGS = 4  # Newey-West lags for daily differences, as in the analysis notebooks.


def window_tags(run_dir: Path) -> list[str]:
    files = sorted((run_dir / "predictions").glob("predictions_*.parquet"))
    return [f.name.removeprefix("predictions_").removesuffix(".parquet") for f in files]


tags = window_tags(RUN_DIR)
assert tags == window_tags(PARTNER_DIR), "the two runs have different windows"


def load(kind: str) -> pl.DataFrame:
    """Rows of both runs side by side: p_run, p_partner, label, window (and fold)."""
    keys = ["Date", "Code", "fold"] if kind == "validation" else ["Date", "Code"]
    frames = []
    for k, tag in enumerate(tags, start=1):
        a = pl.read_parquet(RUN_DIR / "predictions" / f"{kind}_{tag}.parquet")
        b = pl.read_parquet(PARTNER_DIR / "predictions" / f"{kind}_{tag}.parquet")
        both = a.rename({"Prediction": "p_run"}).join(
            b.select(*keys, pl.col("Prediction").alias("p_partner"),
                     pl.col("label").alias("label_partner")), on=keys)
        assert (both["label"] == both["label_partner"]).all(), "labels differ"
        print(f"W{k} {kind}: run {a.height:,} rows, partner {b.height:,}, "
              f"common {both.height:,}")
        frames.append(both.drop("label_partner").with_columns(pl.lit(k).alias("window")))
    return pl.concat(frames)


val = load("validation")
test = load("predictions")


## 2. The transformation

For a weight `w` between 0 and 1:

1. Demean each model's predictions per day: `p − mean of all stocks' p that day`.
   This uses only that day's predictions, not labels.
2. `blend = w × run_demeaned + (1 − w) × partner_demeaned`.

The blend stays demeaned. Because both inputs are in label units, so is the blend.


In [ ]:
def demeaned(column: str) -> pl.Expr:
    """The column minus its mean over all stocks on the same day."""
    return pl.col(column) - pl.col(column).mean().over("Date")


def add_blend(df: pl.DataFrame, w: float, name: str = "blend") -> pl.DataFrame:
    """blend = w x run (demeaned) + (1 - w) x partner (demeaned)."""
    return df.with_columns(
        (w * demeaned("p_run") + (1 - w) * demeaned("p_partner")).alias(name))


def ols_slope(p: np.ndarray, y: np.ndarray) -> float:
    p, y = p - p.mean(), y - y.mean()
    return float(p @ y / (p @ p))


def daily(df: pl.DataFrame, col: str) -> pl.DataFrame:
    """Per day: Rank IC, Pearson IC, Q5-Q1."""
    q = ((pl.col(col).rank() * 5 / pl.len()).ceil().clip(1, 5)).over("Date")
    return (df.with_columns(q.alias("q")).group_by("window", "Date").agg(
        pl.corr(pl.col(col).rank(), pl.col("label").rank()).alias("rank_ic"),
        pl.corr(col, "label").alias("ic"),
        (pl.col("label").filter(pl.col("q") == 5).mean()
         - pl.col("label").filter(pl.col("q") == 1).mean()).alias("q5_q1"),
    ).sort("window", "Date"))


def fold_r2(df: pl.DataFrame, col: str) -> float:
    """Validation R²: mean over folds of 1 - MSE / label variance (as the sweeps)."""
    r2 = [1 - float(((f["label"] - f[col]) ** 2).mean()) / float(f["label"].var(ddof=0))
          for f in df.partition_by("fold")]
    return float(np.mean(r2))


def pooled_r2(df: pl.DataFrame, col: str) -> float:
    """Test R²: pooled over rows, against the overall label mean (as the analyses)."""
    y, p = df["label"].to_numpy(), df[col].to_numpy()
    return 1 - float(((y - p) ** 2).sum()) / float(((y - y.mean()) ** 2).sum())


def newey_west_se(x: np.ndarray, lags: int = NW_LAGS) -> float:
    x = x - x.mean()
    n, v = len(x), float(x @ x) / len(x)
    for lag in range(1, lags + 1):
        v += 2 * (1 - lag / (lags + 1)) * float(x[lag:] @ x[:-lag]) / n
    return float(np.sqrt(v / n))


## 3. Choosing w on validation, per window

For each window and each `w` in 0, 0.05, …, 1: the fold-mean R² and the mean daily
Rank IC of the blend on that window's validation predictions. Each window keeps:
- **w_R²:** the weight with the best validation R² (magnitudes);
- **w_RankIC:** the weight with the best validation Rank IC (ranking).


In [ ]:
rows = []
for k in range(1, len(tags) + 1):
    v = val.filter(pl.col("window") == k)
    for w in WEIGHTS:
        b = add_blend(v, w)
        rows.append({"window": k, "w": w, "val_r2": fold_r2(b, "blend"),
                     "val_rank_ic": daily(b, "blend")["rank_ic"].mean()})
val_scan = pl.DataFrame(rows)
chosen = val_scan.group_by("window").agg(
    pl.col("w").sort_by("val_r2").last().alias("w_R2"),
    pl.col("w").sort_by("val_rank_ic").last().alias("w_RankIC"),
).sort("window")
print("Weight on this run, chosen per window on its validation predictions:")
print(chosen)
print("\nValidation curves (rows: w; columns: window), R²:")
print(val_scan.pivot(on="window", index="w", values="val_r2")
      .with_columns(pl.selectors.float().round(5)))


## 4. Test results with the chosen weights

Each window's test predictions are blended with its own chosen weight. Compared
with the partner alone (w = 0) and this run alone (w = 1), all demeaned.
- **Ranking:** Rank IC, Pearson IC, Q5−Q1.
- **Magnitudes:** pooled R², mean |slope − 1| over windows.
- **ΔRank IC** vs the partner: the paired daily difference, with a Newey–West 95% CI.


In [ ]:
def blend_with(df: pl.DataFrame, weights: dict[int, float]) -> pl.DataFrame:
    """Each window's rows blended with its own weight."""
    return pl.concat([add_blend(df.filter(pl.col("window") == k), w)
                      for k, w in weights.items()])


rules = {
    "partner alone (w = 0)": dict.fromkeys(range(1, len(tags) + 1), 0.0),
    "this run alone (w = 1)": dict.fromkeys(range(1, len(tags) + 1), 1.0),
    "w_R² (per window)": dict(zip(chosen["window"], chosen["w_R2"], strict=True)),
    "w_RankIC (per window)": dict(zip(chosen["window"], chosen["w_RankIC"], strict=True)),
}
base_daily = daily(blend_with(test, rules["partner alone (w = 0)"]), "blend")
summary, per_window = [], []
for name, weights in rules.items():
    b = blend_with(test, weights)
    d = daily(b, "blend")
    diff = (d["rank_ic"] - base_daily["rank_ic"]).to_numpy()
    se = newey_west_se(diff)
    slopes = [ols_slope(b.filter(pl.col("window") == k)["blend"].to_numpy(),
                        b.filter(pl.col("window") == k)["label"].to_numpy())
              for k in weights]
    summary.append({
        "rule": name, "Rank IC": d["rank_ic"].mean(), "ΔRank IC vs partner": diff.mean(),
        "± 95% (NW)": 1.96 * se, "t": diff.mean() / se if se else 0.0,
        "Pearson IC": d["ic"].mean(), "Q5-Q1": d["q5_q1"].mean(),
        "R²": pooled_r2(b, "blend"),
        "mean |slope - 1|": float(np.mean([abs(s - 1) for s in slopes])),
    })
    for k, w in weights.items():
        bw = b.filter(pl.col("window") == k)
        per_window.append({"rule": name, "window": k, "w": w,
                           "Rank IC": d.filter(pl.col("window") == k)["rank_ic"].mean(),
                           "R²": pooled_r2(bw, "blend"), "slope": slopes[k - 1]})
print(pl.DataFrame(summary).with_columns(pl.selectors.float().round(5)))
pw = pl.DataFrame(per_window)
print("\nPer window:")
print(pw.pivot(on="rule", index="window", values=["Rank IC", "R²"])
      .with_columns(pl.selectors.float().round(5)))


## 5. Hindsight scan on test (descriptive only)

The same blend with **one weight for all windows**, scanned over the test sets.
Picking the best point of this curve would use the test labels, so it's shown only
to see how the test responds to w, and how well validation predicted it.


In [ ]:
rows = []
for w in WEIGHTS:
    b = add_blend(test, w)
    d = daily(b, "blend")
    rows.append({"w": w, "Rank IC": d["rank_ic"].mean(), "Pearson IC": d["ic"].mean(),
                 "R²": pooled_r2(b, "blend")})
test_scan = pl.DataFrame(rows)
print(test_scan.with_columns(pl.selectors.float().round(5)))
val_mean = val_scan.group_by("w").agg(pl.col("val_r2").mean(),
                                      pl.col("val_rank_ic").mean()).sort("w")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), layout="constrained")
for ax, (t_col, v_col, title) in zip(axes, [
        ("Rank IC", "val_rank_ic", "Rank IC vs blend weight"),
        ("R²", "val_r2", "R² (demeaned) vs blend weight")], strict=True):
    ax.plot(test_scan["w"], test_scan[t_col], "o-", c="#2a6fdb", lw=2, ms=5,
            label="test (one w for all windows)")
    ax.plot(val_mean["w"], val_mean[v_col], "s--", c="#8c8c8c", lw=1.5, ms=4,
            label="validation (mean over windows)")
    ax.set_title(title)
    ax.set_xlabel(f"w = weight on {RUN_DIR.name.split('_')[0]} "
                  f"(1 - w on {PARTNER_DIR.name.split('_')[0]})")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, frameon=False)
fig.suptitle(f"Blend of {RUN_DIR.name} and {PARTNER_DIR.name}", fontsize=10)
fig.savefig(PLOTS_DIR / "post_blending.png", dpi=110)
plt.show()


## 6. Findings

1. **The two runs line up:** all windows have identical rows and labels, for
   validation (3.8 M rows) and test (2.03 M rows).
2. **The weights chosen on validation vary widely:**
   - w_R² 0.05 / 0.25 / 0.05 / 0.45 / 0.35 / 0.15 (W1–W6);
   - w_RankIC 0.35 / 0.9 / 1.0 / 1.0 / 0.25 / 0.65.
   - Validation favoured run 7 most in W3/W4. Same pattern as in the sweep and
     in run 7's own analysis (N-28, N-29).
3. **Test, honest rules (weights from validation):**

| Rule | Rank IC | ΔRank IC vs run 5 (95% NW) | Pearson IC | R² (demeaned) | Mean \|slope − 1\| |
|---|---|---|---|---|---|
| run 5 alone (w = 0) | **0.0631** | — | **0.0985** | **0.00942** | **0.178** |
| run 7 alone (w = 1) | 0.0629 | −0.0003 ± 0.0044 | 0.0897 | 0.00709 | 0.408 |
| w_R² per window | 0.0625 | −0.0007 ± 0.0012 (t −1.0) | 0.0976 | 0.00925 | 0.202 |
| w_RankIC per window | 0.0605 | −0.0026 ± 0.0031 (t −1.7) | 0.0937 | 0.00781 | 0.356 |

   - Neither rule beats run 5 on any metric.
   - The R² rule costs a little everywhere: −2% R², calibration 0.18 → 0.20.
   - The Rank IC rule repeats run 7's W4 collapse, because validation chose
     w = 1 there. W4 Rank IC 0.0333 vs 0.0571.
4. **Hindsight (one w for all windows, chosen on test; descriptive only):**
   - The best Rank IC is at w = 0.35: 0.06345, i.e. +0.0003 over run 5, within
     seed noise (±0.0004).
   - At that weight R² is 0.00920 (−2%) and Pearson IC 0.0979 (−0.0006).
   - R² is flat up to w ≈ 0.1 and then falls steadily.
   - There is no weight where the blend clearly beats run 5, even with
     hindsight.
5. **Validation vs test:**
   - Validation promised a Rank IC peak near w = 0.75 (+0.0029 over run 5's
     level) with R² flat up to w ≈ 0.4.
   - On test, Rank IC moves by at most +0.0003, and R² declines from
     w = 0.1.
   - Same pattern as run 7 alone: day-context gains in validation don't carry
     over to test (N-29).
6. **Per window,** the R² rule helps where run 7 was good (W5 Rank IC 0.0574 →
   0.0609, R² 0.0044 → 0.0049) and hurts in W4 (0.0571 → 0.0477, R² 0.0066 →
   0.0056). These roughly cancel.


## 7. Summary

- **Blending run 7 into run 5 doesn't help.** With weights chosen on
  validation, the blend is slightly worse than run 5 alone (Rank IC −0.0007,
  R² −2%, calibration worse).
- Even the best single weight in hindsight gains only +0.0003 Rank IC (seed
  noise) at −2% R².
- **Why:**
  - Run 7's extra signal (2025, calm days) and its W4 collapse cancel out.
  - Validation overrates run 7 in exactly the windows where it fails on test
    (W3/W4), so weights picked there backfire.
- **Decision:** keep Run-5 (demeaned) as the reference. Don't adopt run 7,
  alone or blended.
- **Lesson for later:** a blend is worth it only if the second model's gains
  are not driven by regime-identifying inputs that validation overrates.
